# AWS Lambda Functions with Boto3

Region: ap-south-1

In [ ]:
import boto3
import io
import json
import zipfile
from botocore.exceptions import ClientError

REGION = "ap-south-1"
DRY_RUN = True

iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)

In [ ]:
# modify these variables as per your requirements
ROLE_NAME = "exam-lambda-role",
ACCOUNT_ID = "123456789012",
FUNCTION_NAME = "exam-lambda",
RUNTIME = "python3.11"
HANDLER = "lambda_function.lambda_handler"

ASSUME_ROLE_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ],
}

LAMBDA_CODE = [
    "def lambda_handler(event, context):\n"
    "    return {\"statusCode\": 200, \"body\": \"ok\"}\n"
]

In [ ]:
def zip_lambda(code_lines):
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("lambda_function.py", "".join(code_lines))
    return buffer.getvalue()

zip_bytes = zip_lambda(LAMBDA_CODE)

In [ ]:
if DRY_RUN:
    print("[DRY RUN] create_role, attach_role_policy, create_function")
else:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(ASSUME_ROLE_POLICY),
    )
    iam.attach_role_policy(
        RoleName=ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    # Role propagation can take time. Add a short wait if needed.
    lambda_client.create_function(
        FunctionName=FUNCTION_NAME,
        Runtime=RUNTIME,
        Role=f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}",
        Handler=HANDLER,
        Code={"ZipFile": zip_bytes},
        Timeout=10,
    )

In [ ]:
if DRY_RUN:
    print("[DRY RUN] invoke_function, update_function_code")
else:
    payload = json.dumps({"ping": "pong"}).encode("utf-8")
    print(lambda_client.invoke(FunctionName=FUNCTION_NAME, Payload=payload))
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_bytes)

In [ ]:
if DRY_RUN:
    print("[DRY RUN] delete_function, detach_role_policy, delete_role")
else:
    lambda_client.delete_function(FunctionName=FUNCTION_NAME)
    iam.detach_role_policy(
        RoleName=ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    iam.delete_role(RoleName=ROLE_NAME)